# Week 4: QCCA-V2 Diagnostic and Policy-Refinement Study

**Project:** Query-Conditioned Context Allocator (QCCA) for Efficient Retrieval-Augmented Generation  
**Parent Architecture:** SARA — Selective and Adaptive Retrieval-augmented Generation with Context Compression  
**Objective:** Diagnose why the current learned sequential policy fails to convert discovered feature signal into a policy that clearly improves over static-k baselines, and determine what sequential policy structure is actually justified by the data.

### Strict Governance Controls:
1. **Zero Test Set Access:** Development split only (86 papers, 231 queries). The historical test set remains completely frozen.
2. **Paper-Disjoint GroupKFold:** All modeling, scaling, and threshold evaluations are strictly out-of-fold grouped by `paper_id`.
3. **No Feature Leakage:** Only pre-generation observable features are used.


In [1]:
import os
import sys
from pathlib import Path

# Robust project root discovery (supports running from repo root or notebooks/week4)
_curr = Path.cwd().resolve()
candidates = [_curr, _curr.parent, _curr.parent.parent]
PROJECT_ROOT = next((p for p in candidates if (p / 'src').is_dir() and (p / 'results').is_dir()), _curr)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
os.chdir(PROJECT_ROOT)
print(f"Working directory set to Project Root: {PROJECT_ROOT}")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.week4.qcca_v2 import (
    CONTEXT_TOKEN_COSTS,
    STAGE_1_FEATURES,
    STAGE_2_FEATURES,
    STAGE_3_FEATURES,
    PROVISIONAL_14_FEATURES,
    load_qcca_v2_data,
    compute_pareto_frontier,
)
from src.week4.qcca_v2_diagnostic import (
    FEATURE_FAMILIES,
    train_eval_stage_oof_detailed,
    compute_stage_error_consequences,
    simulate_policy_structure,
    sweep_policy_thresholds,
    nested_infold_threshold_selection,
    evaluate_cost_sensitive_rules,
    train_eval_stage_regressors_oof,
    run_stage_feature_family_ablations,
    evaluate_compact_subsets,
    conduct_detailed_oracle_gap_analysis,
    run_policy_paired_bootstrap,
    plot_stage_roc_pr_curves,
    plot_calibration_curves,
    plot_threshold_curves,
    plot_policy_pareto_frontier,
    plot_oracle_gap_and_failures,
    plot_error_propagation_comparison,
    plot_budget_distributions,
)

OUTPUT_DIR = PROJECT_ROOT / "results/week4/qcca_v2_diagnostic"
FIG_DIR = OUTPUT_DIR / "figures"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)

print("Environment initialized successfully. Output paths configured.")


Working directory set to Project Root: /home/gunavenkat/Downloads/CS787-RAG-Project
Environment initialized successfully. Output paths configured.


## 1. Data Loading & Alignment
Loads the 231 QASPER development queries across 86 papers with frozen response surfaces ($k \in \{2, 4, 6, 8\}$) and deployable pre-generation features.


In [2]:
df_f, df_t, df_merged = load_qcca_v2_data(
    feature_matrix_path="results/week4/feature_discovery/feature_matrix.csv",
    dev_matrix_path="results/week2/processed/dev_per_query_k_matrix.csv",
    oracle_path="results/week3/processed/epsilon_oracle_all.csv",
)
print(f"Loaded {len(df_merged)} queries across {df_merged['paper_id'].nunique()} papers.")
print(f"Oracle F1 Mean: {df_merged['oracle_f1'].mean():.4f}, Oracle k Mean: {df_merged['oracle_k'].mean():.2f}")
df_merged[['question_id', 'paper_id', 'F1_k2', 'F1_k4', 'F1_k6', 'F1_k8', 'oracle_k', 'oracle_f1']].head()


Loaded 231 queries across 86 papers.
Oracle F1 Mean: 0.4894, Oracle k Mean: 3.49


,question_id,paper_id,F1_k2,F1_k4,F1_k6,F1_k8,oracle_k,oracle_f1
0,1912,638,0.0000,0.0000,0.0000,0.5000,8,0.5
1,2201,798,0.8000,0.6316,0.0000,0.6316,2,0.8
2,883,254,0.6667,0.6667,1.0000,0.6667,6,1.0
3,884,254,1.0000,1.0000,0.8889,0.8889,2,1.0
4,104,28,0.0000,0.0000,0.0000,0.0000,2,0.0


## 2. Analysis 1 — Stage-Wise Error Diagnosis (2→4, 4→6, 6→8)
Evaluates out-of-fold performance for all three transitions under paper-disjoint GroupKFold cross-validation. Computes ROC-AUC, PR-AUC, Balanced Accuracy, Precision, Recall, Specificity, Brier score, and ECE.


In [3]:
# Evaluate each transition stage out-of-fold
res_s1 = train_eval_stage_oof_detailed(df_merged, "Stage 1 (2->4)", "Y_2_4", STAGE_1_FEATURES, "logistic_regression_balanced")
res_s2 = train_eval_stage_oof_detailed(df_merged, "Stage 2 (4->6)", "Y_4_6", STAGE_2_FEATURES, "logistic_regression_balanced")
res_s3 = train_eval_stage_oof_detailed(df_merged, "Stage 3 (6->8)", "Y_6_8", STAGE_3_FEATURES, "logistic_regression_balanced")

stage_results = {
    "Stage 1 (2->4)": res_s1,
    "Stage 2 (4->6)": res_s2,
    "Stage 3 (6->8)": res_s3,
}

df_stagewise_metrics = pd.DataFrame([
    {k: v for k, v in res.items() if not isinstance(v, (np.ndarray, pd.DataFrame))}
    for res in stage_results.values()
])
df_stagewise_metrics.to_csv(OUTPUT_DIR / "stagewise_oof_metrics.csv", index=False)
display(df_stagewise_metrics)


,stage,model_name,n_samples,prevalence_pos,prevalence_neg,oof_roc_auc,oof_pr_auc,oof_balanced_acc,oof_precision,oof_recall,...,oof_brier_score,oof_ece,tp,fp,tn,fn,fold_roc_mean,fold_roc_std,fold_bal_mean,fold_bal_std
0,Stage 1 (2->4),logistic_regression_balanced,231,0.238095,0.761905,0.4385,0.2101,0.4551,0.2019,0.3818,...,0.2549,0.2541,21,83,93,34,0.5097,0.1179,0.4819,0.1085
1,Stage 2 (4->6),logistic_regression_balanced,231,0.194805,0.805195,0.6503,0.3199,0.6448,0.3053,0.6444,...,0.2307,0.2687,29,66,120,16,0.6557,0.0690,0.6489,0.0319
2,Stage 3 (6->8),logistic_regression_balanced,231,0.160173,0.839827,0.5341,0.2532,0.5229,0.1758,0.4324,...,0.2424,0.3222,16,75,119,21,0.5626,0.0925,0.5310,0.0713


### Downstream Policy Consequences of Stage-Wise Errors
Quantifies the actual answer F1 loss, wasted token cost, and regret resulting from False Expansions vs False Stoppings.


In [4]:
cons_s1 = compute_stage_error_consequences(df_merged, "Stage 1 (2->4)", "Y_2_4", "G_2_4", res_s1["oof_preds"], token_delta=411.6)
cons_s2 = compute_stage_error_consequences(df_merged, "Stage 2 (4->6)", "Y_4_6", "G_4_6", res_s2["oof_preds"], token_delta=413.0)
cons_s3 = compute_stage_error_consequences(df_merged, "Stage 3 (6->8)", "Y_6_8", "G_6_8", res_s3["oof_preds"], token_delta=399.4)

df_confusion_consequences = pd.concat([cons_s1, cons_s2, cons_s3], ignore_index=True)
df_confusion_consequences.to_csv(OUTPUT_DIR / "stagewise_confusion_matrices.csv", index=False)
display(df_confusion_consequences)


,stage,category,query_count,percentage,mean_marginal_gain,total_marginal_gain,token_cost_delta,policy_impact
0,Stage 1 (2->4),TP (Beneficial Expansion),21,9.09,0.3473,7.2936,411.6,Captured +7.29 cumulative F1 gain; consumed +4...
1,Stage 1 (2->4),FP (False Expansion / Wasted Context),83,35.93,-0.0509,-4.2248,411.6,Wasted +411.6 tokens per query; net marginal F...
2,Stage 1 (2->4),FN (False Stopping / Missed Gain),34,14.72,0.3870,13.1580,0.0,Missed opportunity cost: lost 13.16 cumulative...
3,Stage 1 (2->4),TN (True Stopping / Distractor Avoided),93,40.26,-0.0360,-3.3449,0.0,Saved 411.6 tokens per query; avoided net nega...
4,Stage 2 (4->6),TP (Beneficial Expansion),29,12.55,0.3395,9.8448,413.0,Captured +9.84 cumulative F1 gain; consumed +4...
5,Stage 2 (4->6),FP (False Expansion / Wasted Context),66,28.57,-0.0632,-4.1696,413.0,Wasted +413.0 tokens per query; net marginal F...
6,Stage 2 (4->6),FN (False Stopping / Missed Gain),16,6.93,0.3802,6.0828,0.0,Missed opportunity cost: lost 6.08 cumulative ...
7,Stage 2 (4->6),TN (True Stopping / Distractor Avoided),120,51.95,-0.0477,-5.7293,0.0,Saved 413.0 tokens per query; avoided net nega...
8,Stage 3 (6->8),TP (Beneficial Expansion),16,6.93,0.2613,4.1804,399.4,Captured +4.18 cumulative F1 gain; consumed +3...
9,Stage 3 (6->8),FP (False Expansion / Wasted Context),75,32.47,-0.0486,-3.6478,399.4,Wasted +399.4 tokens per query; net marginal F...


## 3. Analysis 2 — Error Propagation & Policy Structure Variations
Explicitly tests whether Stage 1 errors bottleneck the stronger Stage 2 signal by comparing:
- **Policy A:** Standard cascade ($2 \rightarrow 4 \rightarrow 6 \rightarrow 8$)
- **Policy B:** Start at $k=4 \rightarrow 6 \rightarrow 8$ (bypasses Stage 1 entirely)
- **Policy C1:** Start at $k=4 \rightarrow 6$, stop at $k=6$
- **Policy C2:** Start at $k=4 \rightarrow 6$, expand to $k=8$
- **Policy D1:** Conservative Stage 1 ($t_{S1} = 0.60$, high-precision)
- **Policy D2:** High-Recall Stage 1 ($t_{S1} = 0.30$, low barrier)


In [5]:
# Policy A (Standard cascade)
df_res_a, sm_a = simulate_policy_structure(
    df_merged, "A_full_cascade", res_s1["oof_preds"], res_s2["oof_preds"], res_s3["oof_preds"], policy_name="Policy A (2->4->6->8)"
)

# Policy B (Start k=4)
df_res_b, sm_b = simulate_policy_structure(
    df_merged, "B_start_k4", res_s1["oof_preds"], res_s2["oof_preds"], res_s3["oof_preds"], policy_name="Policy B (Start k=4->6->8)"
)

# Policy C1 (Start k=4, stop k=6)
df_res_c1, sm_c1 = simulate_policy_structure(
    df_merged, "C_start_k4_stop_k6", res_s1["oof_preds"], res_s2["oof_preds"], res_s3["oof_preds"], policy_name="Policy C1 (Start k=4, stop k=6)"
)

# Policy C2 (Start k=4, expand k=8)
df_res_c2, sm_c2 = simulate_policy_structure(
    df_merged, "C_start_k4_expand_k8", res_s1["oof_preds"], res_s2["oof_preds"], res_s3["oof_preds"], policy_name="Policy C2 (Start k=4, expand k=8)"
)

# Policy D1 (Conservative S1: t=0.60)
s1_p_d1 = (res_s1["oof_probs"] >= 0.60).astype(int)
df_res_d1, sm_d1 = simulate_policy_structure(
    df_merged, "D_conservative_s1", s1_p_d1, res_s2["oof_preds"], res_s3["oof_preds"], policy_name="Policy D1 (Conservative S1, t=0.60)"
)

# Policy D2 (High-Recall S1: t=0.30)
s1_p_d2 = (res_s1["oof_probs"] >= 0.30).astype(int)
df_res_d2, sm_d2 = simulate_policy_structure(
    df_merged, "D_aggressive_s1", s1_p_d2, res_s2["oof_preds"], res_s3["oof_preds"], policy_name="Policy D2 (High-Recall S1, t=0.30)"
)

df_error_prop = pd.DataFrame([sm_a, sm_b, sm_c1, sm_c2, sm_d1, sm_d2])
df_error_prop.to_csv(OUTPUT_DIR / "error_propagation.csv", index=False)
display(df_error_prop)


,policy,mean_f1,mean_k,mean_context_tokens,context_reduction_pct_vs_k8,f1_per_1k_tokens,mean_regret,median_regret,fraction_zero_regret,pct_k2,pct_k4,pct_k6,pct_k8
0,Policy A (2->4->6->8),0.3318,3.50,842.6,52.08,0.3937,0.1577,0.0,61.90,55.0,19.0,22.1,3.9
1,Policy B (Start k=4->6->8),0.3845,5.00,1152.3,34.47,0.3337,0.1050,0.0,70.56,0.0,58.9,32.0,9.1
2,"Policy C1 (Start k=4, stop k=6)",0.3842,4.82,1115.9,36.54,0.3443,0.1053,0.0,69.70,0.0,58.9,41.1,0.0
3,"Policy C2 (Start k=4, expand k=8)",0.3797,5.65,1280.2,27.20,0.2966,0.1098,0.0,69.26,0.0,58.9,0.0,41.1
4,"Policy D1 (Conservative S1, t=0.60)",0.3088,2.22,579.0,67.07,0.5333,0.1806,0.0,57.58,93.5,2.6,3.5,0.4
5,"Policy D2 (High-Recall S1, t=0.30)",0.3853,4.88,1127.3,35.89,0.3418,0.1042,0.0,70.56,3.5,57.6,30.3,8.7


## 4. Analysis 3 — Decision Threshold Optimization & Nested In-Fold Tuning
Sweeps decision thresholds over $t \in [0.10, 0.90]$ and performs strictly leak-free nested in-fold threshold selection.


In [6]:
# Threshold sweep across grid
df_thresh_sweep = sweep_policy_thresholds(
    df_merged, res_s1["oof_probs"], res_s2["oof_probs"], res_s3["oof_probs"], thresholds=np.linspace(0.10, 0.90, 17)
)
df_thresh_sweep.to_csv(OUTPUT_DIR / "threshold_sensitivity.csv", index=False)
display(df_thresh_sweep)

# Leak-free nested in-fold threshold tuning
df_nested_res, sm_nested, df_nested_sel = nested_infold_threshold_selection(
    df_merged, res_s1["oof_probs"], res_s2["oof_probs"], res_s3["oof_probs"], res_s1["fold_ids"], objective="utility", lambda_param=0.0001
)
print("Nested In-Fold Selected Thresholds per validation fold:")
display(df_nested_sel)
print("Nested Tuned Policy Performance:")
display(pd.DataFrame([sm_nested]))


,policy,mean_f1,mean_k,mean_context_tokens,context_reduction_pct_vs_k8,f1_per_1k_tokens,mean_regret,median_regret,fraction_zero_regret,pct_k2,pct_k4,pct_k6,pct_k8,threshold
0,Threshold_t=0.10,0.3950,7.84,1725.0,1.91,0.2290,0.0945,0.0,70.56,2.2,0.9,0.0,97.0,0.10
1,Threshold_t=0.15,0.3916,7.70,1696.8,3.51,0.2308,0.0979,0.0,69.70,2.2,4.3,0.0,93.5,0.15
2,Threshold_t=0.20,0.3863,7.54,1665.2,5.31,0.2320,0.1032,0.0,69.70,2.2,8.2,0.0,89.6,0.20
3,Threshold_t=0.25,0.3915,7.38,1631.9,7.20,0.2399,0.0980,0.0,71.86,2.2,11.7,1.3,84.8,0.25
4,Threshold_t=0.30,0.3846,7.03,1561.7,11.19,0.2463,0.1048,0.0,70.56,3.5,16.9,4.3,75.3,0.30
5,Threshold_t=0.35,0.3767,6.55,1463.3,16.78,0.2575,0.1127,0.0,68.40,6.1,22.9,8.7,62.3,0.35
6,Threshold_t=0.40,0.3629,5.97,1345.9,23.46,0.2696,0.1266,0.0,66.67,7.8,31.6,15.2,45.5,0.40
7,Threshold_t=0.45,0.3554,5.12,1173.6,33.26,0.3028,0.1341,0.0,63.20,18.6,32.0,24.2,25.1,0.45
8,Threshold_t=0.50,0.3318,3.50,842.6,52.08,0.3937,0.1577,0.0,61.90,55.0,19.0,22.1,3.9,0.50
9,Threshold_t=0.55,0.3180,2.56,650.4,63.02,0.4890,0.1714,0.0,60.17,83.1,6.5,9.5,0.9,0.55


Nested In-Fold Selected Thresholds per validation fold:


,val_fold,selected_threshold,train_best_score
0,0,0.70,0.2510
1,1,0.55,0.2702
2,2,0.50,0.2469
3,3,0.55,0.2462
4,4,0.55,0.2560


Nested Tuned Policy Performance:


,policy,mean_f1,mean_k,mean_context_tokens,context_reduction_pct_vs_k8,f1_per_1k_tokens,mean_regret,median_regret,pct_k2,pct_k4,pct_k6,pct_k8
0,Nested_Tuned_OOF_utility,0.305,2.54,645.1,63.32,0.4729,0.1844,0.0,83.1,7.4,9.1,0.4


## 5. Analysis 4 — Cost-Sensitive & Expected Utility Decision Rules
Investigates whether decision rules based on expected utility ($U = \Delta F_1 - \lambda \times \Delta \text{Tokens}$) outperform flat probability thresholding.


In [7]:
# Train Ridge regressors for continuous gain prediction
reg_s1 = train_eval_stage_regressors_oof(df_merged, "Stage 1", "G_2_4", STAGE_1_FEATURES)
reg_s2 = train_eval_stage_regressors_oof(df_merged, "Stage 2", "G_4_6", STAGE_2_FEATURES)
reg_s3 = train_eval_stage_regressors_oof(df_merged, "Stage 3", "G_6_8", STAGE_3_FEATURES)

df_cost_sensitive = evaluate_cost_sensitive_rules(
    df_merged,
    res_s1["oof_probs"],
    res_s2["oof_probs"],
    res_s3["oof_probs"],
    reg_s1["oof_continuous_preds"],
    reg_s2["oof_continuous_preds"],
    reg_s3["oof_continuous_preds"],
)
display(df_cost_sensitive)


,policy,mean_f1,mean_k,mean_context_tokens,context_reduction_pct_vs_k8,f1_per_1k_tokens,mean_regret,median_regret,fraction_zero_regret,pct_k2,pct_k4,pct_k6,pct_k8
0,Expected_Gain_Threshold_0.01,0.3950,7.97,1753.2,0.30,0.2253,0.0945,0.0,70.56,0.4,0.0,0.0,99.6
1,Utility_Ridge_lambda=0.0e+00,0.3878,6.08,1370.3,22.08,0.2830,0.1017,0.0,71.43,6.9,19.0,37.2,36.8
2,Utility_Ridge_lambda=1.0e-05,0.3789,5.84,1322.7,24.78,0.2864,0.1106,0.0,70.13,9.1,21.6,37.2,32.0
3,Utility_Ridge_lambda=2.5e-05,0.3675,5.45,1241.6,29.39,0.2960,0.1220,0.0,69.26,12.6,26.4,37.2,23.8
4,Utility_Ridge_lambda=5.0e-05,0.3674,4.77,1104.0,37.22,0.3328,0.1220,0.0,68.40,18.2,35.9,35.1,10.8
5,Utility_Ridge_lambda=1.0e-04,0.3376,3.62,867.6,50.66,0.3891,0.1519,0.0,62.77,41.6,39.0,16.5,3.0
6,Utility_Ridge_lambda=2.0e-04,0.3124,2.39,614.7,65.04,0.5081,0.1770,0.0,59.31,83.1,14.3,2.6,0.0
7,Utility_Ridge_lambda=5.0e-04,0.3030,2.04,543.4,69.10,0.5576,0.1864,0.0,57.14,97.8,2.2,0.0,0.0


## 6. Analysis 5 — Direct Gain Regression vs Binary Classification
Compares Continuous Gain Regression (Ridge) against Binary Classification (Balanced Logistic Regression).


In [8]:
_, sm_ridge = simulate_policy_structure(
    df_merged,
    "A_full_cascade",
    (reg_s1["oof_continuous_preds"] > 0.01).astype(int),
    (reg_s2["oof_continuous_preds"] > 0.01).astype(int),
    (reg_s3["oof_continuous_preds"] > 0.01).astype(int),
    policy_name="Ridge Regression (delta=0.01)",
)

df_model_comp = pd.DataFrame([
    {
        "model_type": "Logistic Regression (Balanced)",
        "s1_metric": res_s1["oof_balanced_acc"],
        "s2_metric": res_s2["oof_balanced_acc"],
        "s3_metric": res_s3["oof_balanced_acc"],
        "policy_f1": sm_a["mean_f1"],
        "policy_tokens": sm_a["mean_context_tokens"],
        "policy_regret": sm_a["mean_regret"],
    },
    {
        "model_type": "Ridge Regression (Continuous Gain)",
        "s1_metric": reg_s1["spearman_rho"],
        "s2_metric": reg_s2["spearman_rho"],
        "s3_metric": reg_s3["spearman_rho"],
        "policy_f1": sm_ridge["mean_f1"],
        "policy_tokens": sm_ridge["mean_context_tokens"],
        "policy_regret": sm_ridge["mean_regret"],
    },
])
df_model_comp.to_csv(OUTPUT_DIR / "transition_model_comparison.csv", index=False)
display(df_model_comp)


,model_type,s1_metric,s2_metric,s3_metric,policy_f1,policy_tokens,policy_regret
0,Logistic Regression (Balanced),0.4551,0.6448,0.5229,0.3318,842.6,0.1577
1,Ridge Regression (Continuous Gain),-0.0973,0.1316,0.1268,0.3675,1241.6,0.1220


## 7. Analysis 6 — Feature Family Ablations by Stage
Evaluates individual feature families and Leave-One-Family-Out (LOFO) per transition stage.


In [9]:
abl_s1 = run_stage_feature_family_ablations(df_merged, "Stage 1 (2->4)", "Y_2_4")
abl_s2 = run_stage_feature_family_ablations(df_merged, "Stage 2 (4->6)", "Y_4_6")
abl_s3 = run_stage_feature_family_ablations(df_merged, "Stage 3 (6->8)", "Y_6_8")

df_stage_ablation = pd.concat([abl_s1, abl_s2, abl_s3], ignore_index=True)
df_stage_ablation.to_csv(OUTPUT_DIR / "feature_ablation_by_stage.csv", index=False)

df_fam_summary = df_stage_ablation[df_stage_ablation["ablation_type"].str.startswith("Only")].pivot(
    index="ablation_type", columns="stage", values="roc_auc"
)
df_fam_summary.to_csv(OUTPUT_DIR / "feature_family_ablation.csv")
display(df_fam_summary)


stage,Stage 1 (2->4),Stage 2 (4->6),Stage 3 (6->8)
ablation_type,,,
Only bm25_retrieval,0.5362,0.6501,0.5652
Only evidence_structure,0.5530,0.5597,0.5711
Only lexical_coverage,0.4284,0.6228,0.5687
Only passage_diversity,0.4041,0.6033,0.5294
Only query_complexity,0.4073,0.5350,0.5436
Only semantic_similarity,0.5309,0.5947,0.5375


## 8. Analysis 7 — Check Whether 14-Feature Set is Too Large
Compares compact, highly interpretable subsets (top 1-3 features per stage) against the full 14-feature model.


In [10]:
df_compact = evaluate_compact_subsets(df_merged)
display(df_compact)


,policy,mean_f1,mean_k,mean_context_tokens,context_reduction_pct_vs_k8,f1_per_1k_tokens,mean_regret,median_regret,fraction_zero_regret,pct_k2,pct_k4,pct_k6,pct_k8,s1_bal_acc,s2_bal_acc,s3_bal_acc
0,Full_14_Features,0.3603,3.81,905.3,48.52,0.3980,0.1292,0.0,64.50,51.5,22.9,9.1,16.5,0.5398,0.6172,0.5596
1,Hypothesis_Stage_Specific,0.3318,3.50,842.6,52.08,0.3937,0.1577,0.0,61.90,55.0,19.0,22.1,3.9,0.4551,0.6448,0.5229
2,Top_3_Per_Stage,0.3400,3.45,831.9,52.69,0.4087,0.1494,0.0,63.20,58.0,16.0,21.6,4.3,0.4750,0.6367,0.5603
3,Top_2_Per_Stage,0.3374,3.38,817.9,53.49,0.4125,0.1521,0.0,61.47,61.0,11.7,24.7,2.6,0.5068,0.6038,0.5358
4,Top_1_Per_Stage,0.3163,2.51,639.4,63.64,0.4948,0.1731,0.0,59.74,88.3,1.3,6.9,3.5,0.5188,0.5984,0.5686


## 9. Analysis 8 — Probability Calibration & Reliability Analysis
Evaluates Brier scores and Expected Calibration Errors (ECE) across stages.


In [11]:
df_calib_summary = pd.DataFrame([
    {
        "stage": s_name,
        "brier_score": res["oof_brier_score"],
        "ece": res["oof_ece"],
        "prevalence": res["prevalence_pos"],
    }
    for s_name, res in stage_results.items()
])
df_calib_summary.to_csv(OUTPUT_DIR / "calibration_results.csv", index=False)
display(df_calib_summary)


,stage,brier_score,ece,prevalence
0,Stage 1 (2->4),0.2549,0.2541,0.238095
1,Stage 2 (4->6),0.2307,0.2687,0.194805
2,Stage 3 (6->8),0.2424,0.3222,0.160173


## 10. Analysis 9 — Granular Oracle Gap Analysis & Failure Profiling
Profiles per-query deviations from the oracle ceiling and categorizes failure modes.


In [12]:
df_audit, df_fail_summary = conduct_detailed_oracle_gap_analysis(
    df_merged,
    df_res_a,
    res_s1["oof_preds"],
    res_s2["oof_preds"],
    res_s3["oof_preds"],
    res_s1["oof_probs"],
    res_s2["oof_probs"],
    res_s3["oof_probs"],
)
df_audit.to_csv(OUTPUT_DIR / "oracle_gap_analysis.csv", index=False)
display(df_fail_summary)


,failure_type,query_count,percentage,mean_regret,total_regret,mean_f1_gap
4,Stage 1 Premature Stop (Under-Expansion),34,14.72,0.4826,16.4094,0.4826
2,Other Under-Expansion,25,10.82,0.4041,10.1015,0.4041
5,Stage 2 Over-Expansion (Distractor/Waste),32,13.85,0.1335,4.2731,0.1331
3,Stage 1 Over-Expansion (Distractor/Waste),31,13.42,0.0730,2.2632,0.0730
6,Stage 2 Premature Stop (Under-Expansion),4,1.73,0.4220,1.6881,0.4220
8,Stage 3 Premature Stop (Under-Expansion),6,2.60,0.2157,1.2942,0.2157
7,Stage 3 Over-Expansion (Distractor/Waste),7,3.03,0.0296,0.2069,0.0296
1,Other Over-Expansion,7,3.03,0.0268,0.1878,0.0268
0,Exact Match,85,36.80,0.0000,0.0000,0.0000


## 11. Analysis 10 — Policy Comparison Suite & Multi-Objective Pareto Frontier
Compares candidate sequential policies against Static baselines ($k \in \{2, 4, 6, 8\}$), Oracle Sequential, and Epsilon Oracle.


In [13]:
candidate_policies = [
    sm_a,
    sm_b,
    sm_c1,
    sm_c2,
    sm_d1,
    sm_d2,
    sm_ridge,
    sm_nested,
]

static_baselines = [
    {"policy": "Static k=2", "mean_f1": float(df_merged["F1_k2"].mean()), "mean_k": 2.0, "mean_context_tokens": CONTEXT_TOKEN_COSTS[2], "context_reduction_pct_vs_k8": 69.60, "mean_regret": float(np.mean(np.maximum(0, df_merged["oracle_f1"] - df_merged["F1_k2"]))), "pct_k2": 100.0, "pct_k4": 0.0, "pct_k6": 0.0, "pct_k8": 0.0},
    {"policy": "Static k=4", "mean_f1": float(df_merged["F1_k4"].mean()), "mean_k": 4.0, "mean_context_tokens": CONTEXT_TOKEN_COSTS[4], "context_reduction_pct_vs_k8": 46.20, "mean_regret": float(np.mean(np.maximum(0, df_merged["oracle_f1"] - df_merged["F1_k4"]))), "pct_k2": 0.0, "pct_k4": 100.0, "pct_k6": 0.0, "pct_k8": 0.0},
    {"policy": "Static k=6", "mean_f1": float(df_merged["F1_k6"].mean()), "mean_k": 6.0, "mean_context_tokens": CONTEXT_TOKEN_COSTS[6], "context_reduction_pct_vs_k8": 22.71, "mean_regret": float(np.mean(np.maximum(0, df_merged["oracle_f1"] - df_merged["F1_k6"]))), "pct_k2": 0.0, "pct_k4": 0.0, "pct_k6": 100.0, "pct_k8": 0.0},
    {"policy": "Static k=8", "mean_f1": float(df_merged["F1_k8"].mean()), "mean_k": 8.0, "mean_context_tokens": CONTEXT_TOKEN_COSTS[8], "context_reduction_pct_vs_k8": 0.00, "mean_regret": float(np.mean(np.maximum(0, df_merged["oracle_f1"] - df_merged["F1_k8"]))), "pct_k2": 0.0, "pct_k4": 0.0, "pct_k6": 0.0, "pct_k8": 100.0},
    {"policy": "Oracle Sequential (delta=0.01)", "mean_f1": 0.4069, "mean_k": 2.56, "mean_context_tokens": 650.3, "context_reduction_pct_vs_k8": 63.02, "mean_regret": 0.0825, "pct_k2": 76.2, "pct_k4": 19.9, "pct_k6": 3.5, "pct_k8": 0.4},
    {"policy": "Epsilon Oracle (epsilon=0.01)", "mean_f1": float(df_merged["oracle_f1"].mean()), "mean_k": float(df_merged["oracle_k"].mean()), "mean_context_tokens": 841.3, "context_reduction_pct_vs_k8": 52.16, "mean_regret": 0.0, "pct_k2": 57.1, "pct_k4": 17.3, "pct_k6": 8.2, "pct_k8": 10.0},
]

df_all_policies = pd.DataFrame(candidate_policies + static_baselines)
df_all_policies.to_csv(OUTPUT_DIR / "policy_comparison.csv", index=False)
display(df_all_policies)

df_pareto = compute_pareto_frontier(df_all_policies.rename(columns={"policy": "system"}))
df_pareto.to_csv(OUTPUT_DIR / "policy_pareto_points.csv", index=False)
print("Pareto Optimal Systems:")
display(df_pareto[df_pareto["is_pareto_optimal"]][["system", "mean_f1", "mean_context_tokens", "context_reduction_pct_vs_k8"]])


,policy,mean_f1,mean_k,mean_context_tokens,context_reduction_pct_vs_k8,f1_per_1k_tokens,mean_regret,median_regret,fraction_zero_regret,pct_k2,pct_k4,pct_k6,pct_k8
0,Policy A (2->4->6->8),0.331800,3.500000,842.6,52.08,0.3937,0.157700,0.0,61.90,55.0,19.0,22.1,3.9
1,Policy B (Start k=4->6->8),0.384500,5.000000,1152.3,34.47,0.3337,0.105000,0.0,70.56,0.0,58.9,32.0,9.1
2,"Policy C1 (Start k=4, stop k=6)",0.384200,4.820000,1115.9,36.54,0.3443,0.105300,0.0,69.70,0.0,58.9,41.1,0.0
3,"Policy C2 (Start k=4, expand k=8)",0.379700,5.650000,1280.2,27.20,0.2966,0.109800,0.0,69.26,0.0,58.9,0.0,41.1
4,"Policy D1 (Conservative S1, t=0.60)",0.308800,2.220000,579.0,67.07,0.5333,0.180600,0.0,57.58,93.5,2.6,3.5,0.4
5,"Policy D2 (High-Recall S1, t=0.30)",0.385300,4.880000,1127.3,35.89,0.3418,0.104200,0.0,70.56,3.5,57.6,30.3,8.7
6,Ridge Regression (delta=0.01),0.367500,5.450000,1241.6,29.39,0.2960,0.122000,0.0,69.26,12.6,26.4,37.2,23.8
7,Nested_Tuned_OOF_utility,0.305000,2.540000,645.1,63.32,0.4729,0.184400,0.0,NaN,83.1,7.4,9.1,0.4
8,Static k=2,0.303835,2.000000,534.5,69.60,NaN,0.185556,NaN,NaN,100.0,0.0,0.0,0.0
9,Static k=4,0.359601,4.000000,946.1,46.20,NaN,0.129820,NaN,NaN,0.0,100.0,0.0,0.0


Pareto Optimal Systems:


,system,mean_f1,mean_context_tokens,context_reduction_pct_vs_k8
0,Static k=2,0.303835,534.5,69.60
1,"Policy D1 (Conservative S1, t=0.60)",0.308800,579.0,67.07
3,Oracle Sequential (delta=0.01),0.406900,650.3,63.02
4,Epsilon Oracle (epsilon=0.01),0.489391,841.3,52.16


## 12. Paper-Clustered Paired Bootstrap Hypothesis Testing ($B = 2,000$)
Tests whether candidate policies achieve statistically significant quality differences relative to Static $k \in \{4, 6, 8\}$.


In [14]:
policy_dfs_dict = {
    "Policy A (2->4->6->8)": df_res_a,
    "Policy B (Start k=4)": df_res_b,
    "Policy C1 (Start k=4, stop k=6)": df_res_c1,
    "Policy D1 (Conservative S1)": df_res_d1,
}

df_boot = run_policy_paired_bootstrap(
    df_merged,
    policy_dfs_dict,
    baseline_ks=[4, 6, 8],
    n_bootstrap=2000,
    seed=42,
)
df_boot.to_csv(OUTPUT_DIR / "policy_bootstrap_comparison.csv", index=False)
display(df_boot)


,policy,baseline,mean_delta_f1,ci_95_f1_low,ci_95_f1_high,significant_f1_diff,mean_delta_tokens,ci_95_tok_low,ci_95_tok_high
0,Policy A (2->4->6->8),Static k=4,-0.0278,-0.0616,0.0068,False,-103.5,-166.9,-37.4
1,Policy A (2->4->6->8),Static k=6,-0.0539,-0.0922,-0.0213,True,-516.5,-578.9,-453.5
2,Policy A (2->4->6->8),Static k=8,-0.0632,-0.1044,-0.0248,True,-915.9,-979.2,-853.3
3,Policy B (Start k=4),Static k=4,0.0249,-0.0004,0.0520,False,206.2,160.3,247.4
4,Policy B (Start k=4),Static k=6,-0.0012,-0.0221,0.0204,False,-206.8,-249.7,-165.7
5,Policy B (Start k=4),Static k=8,-0.0105,-0.0419,0.0207,False,-606.2,-651.9,-564.7
6,"Policy C1 (Start k=4, stop k=6)",Static k=4,0.0246,-0.0006,0.0508,False,169.8,135.7,202.4
7,"Policy C1 (Start k=4, stop k=6)",Static k=6,-0.0015,-0.0217,0.0182,False,-243.2,-275.3,-211.2
8,"Policy C1 (Start k=4, stop k=6)",Static k=8,-0.0108,-0.0441,0.0211,False,-642.6,-674.7,-611.2
9,Policy D1 (Conservative S1),Static k=4,-0.0508,-0.0878,-0.0172,True,-367.1,-393.0,-336.7


## 13. Publication-Grade Figures Generation
Generates all diagnostic figures into `results/week4/qcca_v2_diagnostic/figures/`.


In [15]:
print("Generating publication-grade diagnostic figures...")
plot_stage_roc_pr_curves(stage_results, df_merged, str(FIG_DIR / "stage_roc_pr_curves.png"))
plot_calibration_curves(stage_results, str(FIG_DIR / "calibration_curves.png"))
plot_threshold_curves(df_thresh_sweep, str(FIG_DIR / "threshold_vs_f1.png"), str(FIG_DIR / "threshold_vs_token_cost.png"))
plot_policy_pareto_frontier(df_pareto, str(FIG_DIR / "policy_pareto_frontier.png"))
plot_oracle_gap_and_failures(df_fail_summary, str(FIG_DIR / "oracle_gap_breakdown.png"))
plot_error_propagation_comparison(pd.DataFrame([sm_a, sm_b, sm_c1, sm_c2, sm_d1, sm_d2]), str(FIG_DIR / "error_propagation.png"))
plot_budget_distributions(pd.DataFrame([sm_a, sm_b, sm_c1, sm_d1, sm_d2, sm_ridge]), str(FIG_DIR / "budget_distributions.png"))

print(f"All 9 figures saved successfully to {FIG_DIR}")


Generating publication-grade diagnostic figures...


/home/gunavenkat/Downloads/CS787-RAG-Project/src/week4/qcca_v2_diagnostic.py:1238: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `y` variable to `hue` and set `legend=False` for the same effect.

  sns.barplot(data=df_fail_summary, y="failure_type", x="percentage", ax=axes[0], palette="Blues_r")
/home/gunavenkat/Downloads/CS787-RAG-Project/src/week4/qcca_v2_diagnostic.py:1244: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `y` variable to `hue` and set `legend=False` for the same effect.

  sns.barplot(data=df_fail_summary, y="failure_type", x="total_regret", ax=axes[1], palette="Reds_r")
/home/gunavenkat/Downloads/CS787-RAG-Project/src/week4/qcca_v2_diagnostic.py:1263: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.


All 9 figures saved successfully to /home/gunavenkat/Downloads/CS787-RAG-Project/results/week4/qcca_v2_diagnostic/figures


## 14. Diagnostic Study Completion Summary


In [16]:
print("=" * 80)
print("QCCA-V2 DIAGNOSTIC STUDY COMPLETE")
print("=" * 80)
print(f"Artifacts successfully written to: {OUTPUT_DIR}")
print("Generated CSVs:")
for p in sorted(OUTPUT_DIR.glob("*.csv")):
    print(f"  - {p.name}")
print("\nGenerated Figures:")
for p in sorted(FIG_DIR.glob("*.png")):
    print(f"  - {p.name}")
print("=" * 80)


QCCA-V2 DIAGNOSTIC STUDY COMPLETE
Artifacts successfully written to: /home/gunavenkat/Downloads/CS787-RAG-Project/results/week4/qcca_v2_diagnostic
Generated CSVs:
  - calibration_results.csv
  - error_propagation.csv
  - feature_ablation_by_stage.csv
  - feature_family_ablation.csv
  - oracle_gap_analysis.csv
  - policy_bootstrap_comparison.csv
  - policy_comparison.csv
  - policy_pareto_points.csv
  - stagewise_confusion_matrices.csv
  - stagewise_oof_metrics.csv
  - threshold_sensitivity.csv
  - transition_model_comparison.csv

Generated Figures:
  - budget_distributions.png
  - calibration_curves.png
  - error_propagation.png
  - oracle_gap_breakdown.png
  - policy_pareto_frontier.png
  - stage_roc_pr_curves.png
  - threshold_vs_f1.png
  - threshold_vs_token_cost.png
